# Fink/LSST — Download Object-Level Summary (`api/v1/objects`)

Ce notebook complète `01_fink_block_lightcurves.ipynb` en téléchargeant les données
de l'endpoint **`api/v1/objects`** pour tous les `diaObjectId` effectivement
téléchargés dans le notebook 01 (ceux présents dans `lc_cache`, reflétés dans
`flatness_metrics.csv`).

## Pourquoi cet endpoint ?

L'endpoint `/api/v1/objects` retourne des colonnes **agrégées au niveau objet** :
moyennes, médianes, statistiques globales calculées sur l'ensemble des diaSources.
Ces colonnes sont **distinctes** de celles retournées par `/api/v1/sources` (par-détection)
et `/api/v1/fp` (forced photometry par-époque).

La fonction `fetch_objects()` était déjà définie dans le notebook 01 mais **jamais
appelée** dans le workflow. Ce notebook l'exploite pour tous les objets.

## Source canonique des objets

`flatness_metrics.csv` est la trace directe de `lc_cache` (section 7 du notebook 01) :
il ne contient que les objets pour lesquels une light curve a été effectivement
téléchargée avec succès. C'est donc la source la plus fiable pour récupérer
la liste exacte des `diaObjectId` à interroger.

```
flatness_metrics.csv  →  diaObjectId uniques (objets réellement dans lc_cache)
    ↓
POST https://api.lsst.fink-portal.org/api/v1/objects   →  summary par diaObjectId
    ↓
data_FINK_BLOCK_LC_01/objects_all.parquet   (+ .csv)
```


## 1. Imports & configuration

In [ ]:
import requests
import pandas as pd
import numpy as np
import os
import time
import warnings

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
# ── API base URL ──────────────────────────────────────────────────────────────
FINK_API = "https://api.lsst.fink-portal.org"

# ── Répertoire de données (identique au notebook 01) ─────────────────────────
NB_TAG = "FINK_BLOCK_LC_01"
DIR_DATA = f"data_{NB_TAG}"

# Fichier source canonique : trace directe de lc_cache dans le notebook 01
FLATNESS_CSV = os.path.join(DIR_DATA, "flatness_metrics.csv")

# ── Délai entre requêtes API (secondes) ───────────────────────────────────────
API_SLEEP = 0.25

# ── Nombre maximum d'objets (None = tous ; mettre ex. 10 pour un test rapide) ─
MAX_OBJECTS = None

os.makedirs(DIR_DATA, exist_ok=True)
print(f"Data directory : {os.path.abspath(DIR_DATA)}")
print(f"Source CSV     : {os.path.abspath(FLATNESS_CSV)}")
assert os.path.exists(FLATNESS_CSV), f"{FLATNESS_CSV} introuvable — exécuter d'abord le notebook 01."

## 2. API wrapper — `fetch_objects`

In [ ]:
def _post_json(url, payload, timeout=90):
    r = requests.post(url, json=payload, timeout=timeout)
    r.raise_for_status()
    return r.json()


def fetch_objects(diaObjectId, columns=None) -> pd.DataFrame:
    """
    Fetch object-level summary for one diaObjectId via /api/v1/objects.

    Returns a one-row DataFrame with all object-level columns returned by the API,
    or an empty DataFrame if the object is not found or the request fails.
    """
    payload = {"diaObjectId": str(diaObjectId), "output-format": "json"}
    if columns:
        payload["columns"] = columns
    try:
        raw = _post_json(f"{FINK_API}/api/v1/objects", payload)
        return pd.DataFrame(raw) if raw else pd.DataFrame()
    except requests.exceptions.HTTPError as e:
        print(f"  HTTP error for {diaObjectId}: {e}")
        return pd.DataFrame()
    except Exception as e:
        print(f"  Error for {diaObjectId}: {e}")
        return pd.DataFrame()


print("fetch_objects() défini.")

## 3. Charger les `diaObjectId` depuis `flatness_metrics.csv`

`flatness_metrics.csv` est la trace directe de `lc_cache` dans le notebook 01 :
il ne contient **que** les objets pour lesquels une light curve a été téléchargée
avec succès (section 7 du notebook 01). C'est exactement la même liste d'objets
que ceux sauvegardés dans les fichiers parquet `*_src.parquet` et `*_fp.parquet`.

In [ ]:
df_flat = pd.read_csv(FLATNESS_CSV)

# Un objet peut apparaître plusieurs fois (une ligne par bande)
# On prend une ligne par diaObjectId, en gardant le groupe de classification
df_oids = df_flat[["diaObjectId", "group"]].drop_duplicates(subset="diaObjectId").reset_index(drop=True)
df_oids["diaObjectId"] = df_oids["diaObjectId"].astype(str)

print(f"Fichier source     : {FLATNESS_CSV}")
print(f"Objets uniques     : {len(df_oids)}  (issus de lc_cache du notebook 01)")
print(f"\nDistribution par groupe :")
print(df_oids["group"].value_counts().to_string())

if MAX_OBJECTS is not None:
    df_oids = df_oids.head(MAX_OBJECTS)
    print(f"\n[Limité à {MAX_OBJECTS} objets pour le test]")

## 4. Sonder l'API : colonnes disponibles dans `/api/v1/objects`

On interroge un seul objet pour afficher le schema complet.

In [ ]:
sample_oid = df_oids["diaObjectId"].iloc[0]
print(f"Sondage avec diaObjectId = {sample_oid}")

df_sample = fetch_objects(sample_oid)

if df_sample.empty:
    print("ATTENTION : Aucune donnée retournée — vérifier que l'API /api/v1/objects est accessible.")
else:
    print(f"\nColonnes disponibles ({len(df_sample.columns)}) :")
    for col in sorted(df_sample.columns):
        val = df_sample[col].iloc[0]
        print(f"  {col:55s}  {val}")

## 5. Téléchargement de tous les objets

In [ ]:
objects_list = []
n_total = len(df_oids)
n_ok = 0
n_empty = 0

print(f"Téléchargement de {n_total} objets via /api/v1/objects ...")
print("=" * 75)

for i, row in enumerate(df_oids.itertuples(index=False)):
    oid = str(row.diaObjectId)
    group = row.group

    df_obj = fetch_objects(oid)

    if df_obj.empty:
        n_empty += 1
        status = "EMPTY"
    else:
        # Ajouter les colonnes de contexte issues du notebook 01
        df_obj.insert(0, "diaObjectId", oid)
        df_obj.insert(1, "group", group)
        objects_list.append(df_obj)
        n_ok += 1
        status = f"OK  ncols={len(df_obj.columns)}"

    # Afficher la progression toutes les 10 itérations
    if (i + 1) % 10 == 0 or i == 0 or (i + 1) == n_total:
        print(f"  [{i + 1:4d}/{n_total}] {oid}  group={group:35s}  {status}")

    time.sleep(API_SLEEP)

print("\n" + "=" * 75)
print(f"OK={n_ok}   vide={n_empty}")

## 6. Assembler et inspecter le DataFrame global

In [ ]:
if not objects_list:
    raise RuntimeError("Aucun objet téléchargé avec succès — vérifier l'API.")

df_objects = pd.concat(objects_list, ignore_index=True)

# Supprimer les éventuelles colonnes diaObjectId dupliquées retournées par l'API
dup_cols = [c for c in df_objects.columns if c != "diaObjectId" and c.lower() == "diaobjectid"]
if dup_cols:
    df_objects.drop(columns=dup_cols, inplace=True)

print(f"DataFrame final : {len(df_objects)} lignes × {len(df_objects.columns)} colonnes")
print(f"Objets uniques  : {df_objects['diaObjectId'].nunique()}")
print("\nDtypes :")
print(df_objects.dtypes.to_string())

In [ ]:
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 220)
df_objects.head(5)

In [ ]:
df_objects.describe(include="all").T

## 7. Inventaire des colonnes — taux de remplissage

In [ ]:
n = len(df_objects)
fill_rate = df_objects.notna().sum().rename("n_non_null").to_frame()
fill_rate["fill_pct"] = (fill_rate["n_non_null"] / n * 100).round(1)
fill_rate["dtype"] = df_objects.dtypes
fill_rate = fill_rate.sort_values("fill_pct", ascending=False)

print(f"Inventaire des colonnes ({len(fill_rate)}) :\n")
print(fill_rate.to_string())

## 8. Distribution par groupe

In [ ]:
print("Objets par groupe de classification (issus du notebook 01) :")
print(df_objects.groupby("group")["diaObjectId"].count().sort_values(ascending=False).to_string())

## 9. Sauvegarde

In [ ]:
# ── Parquet (format principal, préserve les types) ────────────────────────────
path_parquet = os.path.join(DIR_DATA, "objects_all.parquet")
df_objects.to_parquet(path_parquet, index=False)
print(f"Sauvegardé : {path_parquet}  ({os.path.getsize(path_parquet) / 1e6:.2f} MB)")

# ── CSV (pour consultation rapide / compatibilité) ────────────────────────────
path_csv = os.path.join(DIR_DATA, "objects_all.csv")
df_objects.to_csv(path_csv, index=False)
print(f"Sauvegardé : {path_csv}  ({os.path.getsize(path_csv) / 1e6:.2f} MB)")

print("\nDone.")

## 10. Vérification de la relecture

In [ ]:
df_check = pd.read_parquet(path_parquet)
print(f"Relecture OK : {len(df_check)} lignes × {len(df_check.columns)} colonnes")
print(f"diaObjectId uniques : {df_check['diaObjectId'].nunique()}")
print(f"Colonnes : {df_check.columns.tolist()}")
df_check.head(3)